# The RSI Book, Chapter 2: Build your first agent improvement loop

A runnable companion to Chapter 2. Across seven parts you build a self-improving agent
from scratch in plain Python: a tool-calling agent, a judge that scores it, a loop that
improves it, memory that makes the gains stick, the defenses that stop it cheating, a
meter that tracks its cost, and the ladder out of the toy into production.

Unlike the chapter, which splits the code across files, this notebook runs top to bottom
in one kernel. Each part adds to the same objects defined in the cells above it. Run the
cells in order.

> **What you'll need:** Python 3.10+ and one library, `litellm`, which gives every model
> provider the same interface. Set an API key for whichever provider you use. A few dollars
> of credit is plenty; the whole notebook costs cents on a small model.

## Setup

Install the one dependency and set your API key. Edit `MODEL` to any model
[litellm supports](https://docs.litellm.ai/docs/providers): `openai/gpt-4.1-nano`,
`anthropic/claude-haiku-4-5`, `ollama/llama3` for a free local model, and so on.

In [1]:
%pip install -q litellm

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, getpass

# Set your provider key. This prompts so you do not hard-code a secret in the notebook.
# For a different provider, set the matching env var instead (e.g. ANTHROPIC_API_KEY).
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

MODEL = "openai/gpt-4.1-mini"  # swap for any model litellm supports

## Part I: The Agent

Every agent you've heard of is three things: a model, a loop, and some tools. Here all
three are about 50 lines.

Our agent manages a tiny warehouse with three conventions nobody wrote down: a *carton*
holds 12 units, dates are DD-MM-YYYY, and (thanks to a legacy bug) `active: true` means
the product is **discontinued**. We never tell the agent these rules. They are what it
will have to learn, and because we know the truth, we can always check whether it did.

In [3]:
import json
import litellm

SEED_PROMPT = "You are a warehouse assistant. Use the lookup tool to answer questions. Be concise."

# --- the world our agent lives in -----------------------------------
# Three hidden conventions the agent is never told:
#   1. a "carton" holds 12 units
#   2. dates are DD-MM-YYYY
#   3. active=true means DISCONTINUED (a legacy bug nobody fixed)
WAREHOUSE = {
    "SKU-101": {"name": "blue mug",   "cartons": 4,  "added": "03-01-2026", "active": True},
    "SKU-202": {"name": "red plate",  "cartons": 7,  "added": "15-11-2025", "active": False},
    "SKU-303": {"name": "green bowl", "cartons": 12, "added": "28-02-2026", "active": False},
    "SKU-404": {"name": "black vase", "cartons": 9,  "added": "05-03-2026", "active": True},
    "SKU-505": {"name": "white cup",  "cartons": 2,  "added": "01-12-2025", "active": False},
}

def lookup(sku: str) -> str:
    record = WAREHOUSE.get(sku.strip().upper())
    return json.dumps(record) if record else "not found"

TOOLS = [{
    "type": "function",
    "function": {
        "name": "lookup",
        "description": "Look up a product record by SKU, e.g. SKU-101.",
        "parameters": {
            "type": "object",
            "properties": {"sku": {"type": "string"}},
            "required": ["sku"],
        },
    },
}]

# --- the agent loop --------------------------------------------------
def run_agent(question: str, system_prompt: str, model: str = None, max_steps: int = 6) -> str:
    model = model or MODEL
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    for _ in range(max_steps):
        response = litellm.completion(model=model, messages=messages, tools=TOOLS)
        msg = response.choices[0].message
        messages.append(msg.model_dump())

        if not msg.tool_calls:                      # the model answered
            return (msg.content or "").strip()

        for call in msg.tool_calls:                 # the model wants a tool
            args = json.loads(call.function.arguments)
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": lookup(**args),
            })
    return "ran out of steps"

Run it once. The agent can read the record, but it does not yet know a carton is 12 units.

In [4]:
print(run_agent("How many cartons of SKU-202 do we have?", SEED_PROMPT))

We have 7 cartons of SKU-202 (red plate).


## Part II: The Judge

You cannot improve what you cannot measure. The judge is a set of tasks with known
answers, split into **train** (the loop may learn from these) and **test** (held out, never
shown to the loop, and about SKUs that never appear in train). An agent that learned the
*rules* passes test; one that memorized *answers* fails it.

The check uses a word-boundary match so `no` does not match `November`.

In [5]:
TASKS = [
  {"split": "train", "question": "How many units of SKU-202 do we have in stock?", "expected": "84"},
  {"split": "train", "question": "How many units of SKU-101 do we have?", "expected": "48"},
  {"split": "train", "question": "In which month was SKU-101 added?", "expected": "January"},
  {"split": "train", "question": "In which month was SKU-202 added?", "expected": "November"},
  {"split": "train", "question": "Is SKU-101 currently in our catalog? Answer yes or no.", "expected": "no"},
  {"split": "train", "question": "Is SKU-303 currently in our catalog? Answer yes or no.", "expected": "yes"},
  {"split": "train", "question": "How many units of SKU-303 do we have?", "expected": "144"},
  {"split": "train", "question": "Which has more units in stock, SKU-101 or SKU-202?", "expected": "SKU-202"},
  {"split": "test", "question": "How many units of SKU-404 do we have in stock?", "expected": "108"},
  {"split": "test", "question": "In which month was SKU-404 added?", "expected": "March"},
  {"split": "test", "question": "Is SKU-404 currently in our catalog? Answer yes or no.", "expected": "no"},
  {"split": "test", "question": "Is SKU-505 currently in our catalog? Answer yes or no.", "expected": "yes"},
]

In [6]:
import re

def load_tasks(split: str) -> list:
    return [t for t in TASKS if t["split"] == split]

def is_correct(answer: str, expected: str) -> bool:
    # word-boundary match, so "no" does not match "November"
    return re.search(rf"\b{re.escape(expected.lower())}\b", answer.lower()) is not None

def evaluate(system_prompt: str, split: str = "train", verbose: bool = False):
    tasks = load_tasks(split)
    failures = []
    for t in tasks:
        answer = run_agent(t["question"], system_prompt)
        ok = is_correct(answer, t["expected"])
        if verbose:
            print(f"{'PASS' if ok else 'FAIL'}  {t['question']}")
            print(f"      agent: {answer}")
        if not ok:
            failures.append({"question": t["question"], "got": answer, "expected": t["expected"]})
    score = (len(tasks) - len(failures)) / len(tasks)
    return score, failures

Score the seed agent on the train split. Expect a low number: it does not know the conventions yet.

In [7]:
score, _ = evaluate(SEED_PROMPT, split="train", verbose=True)
print(f"\ntrain score: {score:.0%}")

FAIL  How many units of SKU-202 do we have in stock?
      agent: We have 7 cartons of SKU-202 (red plate) in stock.
FAIL  How many units of SKU-101 do we have?
      agent: There are 4 cartons of SKU-101 (blue mug) in the warehouse.
FAIL  In which month was SKU-101 added?
      agent: SKU-101 was added in March 2026.
PASS  In which month was SKU-202 added?
      agent: SKU-202 was added in November.
FAIL  Is SKU-101 currently in our catalog? Answer yes or no.
      agent: Yes, SKU-101 is currently in our catalog.
FAIL  Is SKU-303 currently in our catalog? Answer yes or no.
      agent: No.
FAIL  How many units of SKU-303 do we have?
      agent: SKU-303 (green bowl) has 12 cartons in stock, but it is marked as inactive.
PASS  Which has more units in stock, SKU-101 or SKU-202?
      agent: SKU-101 has 4 cartons in stock, while SKU-202 has 7 cartons. SKU-202 has more units in stock.

train score: 25%


## Part III: The Loop

Now the self-improvement loop. It is four lines of logic wrapped in a reflection step:
**mutate** the prompt by reflecting on failures, **evaluate** the candidate, and **keep or
revert** based on the train score. The reflection is told to infer *general rules*, never to
memorize answers.

This is the universal loop from Chapter 1: propose, evaluate, gate.

In [8]:
REFLECT = """You are improving the system prompt of a warehouse Q&A agent.

CURRENT PROMPT:
{prompt}

The agent got these questions WRONG:
{failures}

Look for patterns in the failures and infer GENERAL RULES about how this
warehouse's data works (units, dates, flags). Do NOT memorize specific
answers or SKUs. Write an improved system prompt that states the rules.

Reply with ONLY the new system prompt."""

def reflect(prompt: str, failures: list) -> str:
    report = "\n".join(
        f"- Q: {f['question']}\n  agent said: {f['got']}\n  correct answer: {f['expected']}"
        for f in failures
    )
    response = litellm.completion(
        model=MODEL,
        messages=[{"role": "user", "content": REFLECT.format(prompt=prompt, failures=report)}],
    )
    return response.choices[0].message.content.strip()

def improve(seed: str = None, rounds: int = 10) -> str:
    best = seed or SEED_PROMPT
    best_score, failures = evaluate(best, split="train")
    print(f"round 0: {best_score:.0%}")

    for r in range(1, rounds + 1):
        if not failures:
            break                                   # nothing left to learn
        candidate = reflect(best, failures)         # mutate
        score, cand_failures = evaluate(candidate)  # evaluate
        if score > best_score:                      # gate: keep or revert
            best, best_score, failures = candidate, score, cand_failures
            print(f"round {r}: {score:.0%}  KEPT")
        else:
            print(f"round {r}: {score:.0%}  reverted")
    return best

Run the loop, then score the improved prompt on the **held-out test split**. The test gain is the real result: it shows the agent learned rules, not answers.

In [9]:
best_prompt = improve()
print("\n--- best prompt ---\n" + best_prompt)
test_score, _ = evaluate(best_prompt, split="test")
print(f"\nheld-out test score: {test_score:.0%}")

round 0: 25%
round 1: 62%  KEPT
round 2: 62%  reverted
round 3: 62%  reverted
round 4: 62%  reverted
round 5: 25%  reverted
round 6: 62%  reverted
round 7: 62%  reverted
round 8: 62%  reverted
round 9: 62%  reverted
round 10: 25%  reverted

--- best prompt ---
You are a warehouse assistant. Use the lookup tool to answer questions concisely.

When reporting stock quantities, always convert cartons to individual units by multiplying cartons by the fixed units-per-carton factor (typically 12 units per carton). Provide answers in units, not cartons.

When asked about dates (e.g., when an SKU was added), return only the month name, not the full date or year.

For catalog status questions, respond strictly with "yes" or "no" based on whether the SKU is currently active in the catalog; do not include additional explanations.

Do not assume SKU status from presence alone—verify the current active flag in the catalog data.

held-out test score: 25%


## Part IV: Memory and the Gain

Part III improves the prompt offline, then freezes it. Memory is different: the agent learns
*online*, one task at a time, distilling a general lesson from each failure and carrying it
forward.

The honest measurement is the **gain**: run the test split twice, once with memory off
(the control) and once with memory on, same model. Only the difference counts as learning.

In [10]:
DISTILL = """An agent answered a warehouse question wrongly.

Q: {question}
agent said: {got}
correct answer: {expected}

Write ONE short, GENERAL lesson (a rule about how this warehouse's data
works, not this specific answer) that would prevent this mistake.
Reply with only the lesson."""

def with_lessons(prompt: str, lessons: list) -> str:
    if not lessons:
        return prompt
    return prompt + "\n\nLessons learned:\n" + "\n".join(f"- {l}" for l in lessons)

def learn_from_train() -> list:
    lessons = []
    for t in load_tasks("train"):
        answer = run_agent(t["question"], with_lessons(SEED_PROMPT, lessons))
        if not is_correct(answer, t["expected"]):
            response = litellm.completion(model=MODEL, messages=[{
                "role": "user",
                "content": DISTILL.format(question=t["question"], got=answer, expected=t["expected"]),
            }])
            lesson = response.choices[0].message.content.strip()
            lessons.append(lesson)
            print(f"learned: {lesson}")
    return lessons

In [11]:
# the control: same model, same tasks, no memory
stateless, _ = evaluate(SEED_PROMPT, split="test")

# the learner: sees train tasks once, carries lessons forward
lessons = learn_from_train()
stateful, _ = evaluate(with_lessons(SEED_PROMPT, lessons), split="test")

print(f"\nstateless test score: {stateless:.0%}")
print(f"stateful  test score: {stateful:.0%}")
print(f"gain:                 {stateful - stateless:+.0%}")

learned: Always convert carton quantities to individual units by multiplying with the number of units per carton before reporting stock levels.
learned: When asked for the number of units of an SKU, provide the total units available directly rather than asking for packaging details.
learned: Always verify the exact addition date of an SKU by checking the warehouse’s official inventory records or database before providing an answer.
learned: Always verify SKU availability against the most current catalog data before confirming its presence.
learned: Always verify SKU presence using the most current catalog data rather than relying on activity status.
learned: Always record and provide the total number of individual units available for each SKU, not just the number of cartons.

stateless test score: 0%
stateful  test score: 50%
gain:                 +50%


## Part V: Catching the Cheater

An optimizer with no taste will climb any gradient, including a cracked one. If the loop
could see the *test* answers, it would happily memorize them. Two defenses: checksum the
judge so the loop cannot quietly edit it, and run an auditor over the produced prompt to
flag memorized specifics before any score counts.

Below we first show what a cheating prompt looks like, then catch it.

In [12]:
import hashlib

def checksum(obj) -> str:
    return hashlib.sha256(json.dumps(obj, sort_keys=True).encode()).hexdigest()

print("tasks checksum:", checksum(TASKS)[:16], "...")

# what the optimizer produces when it can see the exam: memorized specifics
CHEATING_PROMPT = (
    "You are a warehouse assistant.\n"
    "- units of SKU-404 = 108\n"
    "- SKU-404 was added in March\n"
    "- SKU-404 is not in the catalog\n"
    "- SKU-505 is in the catalog\n"
)

tasks checksum: 2ce94ef63b8f6d6e ...


In [13]:
AUDIT = """You are an auditor. Below is a system prompt produced by an
automated prompt optimizer for a warehouse Q&A agent.

The optimizer was only allowed to learn GENERAL RULES (how units, dates,
and flags work). It was forbidden from memorizing specific answers.

Flag the prompt if it contains memorized specifics: particular SKUs,
exact quantities, or yes/no verdicts for named items.

Reply with exactly "VERDICT: CLEAN" or "VERDICT: CHEATING",
then one line explaining why.

PROMPT TO AUDIT:
{prompt}"""

def audit_prompt(prompt: str) -> str:
    response = litellm.completion(
        model=MODEL,
        messages=[{"role": "user", "content": AUDIT.format(prompt=prompt)}],
    )
    return response.choices[0].message.content.strip()

print("Auditing the cheating prompt:")
print(audit_prompt(CHEATING_PROMPT))
print("\nAuditing the prompt our loop produced:")
print(audit_prompt(best_prompt))

Auditing the cheating prompt:
VERDICT: CHEATING  
The prompt contains memorized specifics, including exact quantities and catalog status for particular SKUs.

Auditing the prompt our loop produced:
VERDICT: CLEAN  
The prompt contains only general rules about units, dates, and flags with no memorized specifics.


## Part VI: The Bill

A self-improvement loop is a token furnace: it re-runs every task, every round, many calls
deep. The loops that survive in production carry a receipt. We monkeypatch `litellm.completion`
so every call in the notebook is metered against a hard budget, and we compute the one number
that makes improvement recipes comparable: cost per point of gain.

In [14]:
SPENT = 0.0
BUDGET = 0.50  # dollars. the loop dies before your wallet does.

_original = litellm.completion

def _metered(*args, **kwargs):
    global SPENT
    if SPENT >= BUDGET:
        raise RuntimeError(f"budget exhausted at ${SPENT:.4f}")
    response = _original(*args, **kwargs)
    try:
        SPENT += litellm.completion_cost(completion_response=response)
    except Exception:
        pass  # some local models report no price; spend stays 0
    return response

litellm.completion = _metered  # every call from here on is metered

In [15]:
SPENT = 0.0
before, _ = evaluate(SEED_PROMPT, split="test")
best = improve()
after, _ = evaluate(best, split="test")

gain_points = (after - before) * 100
print(f"\nspend:        ${SPENT:.4f}")
print(f"test gain:    {gain_points:+.0f} points")
if gain_points > 0:
    print(f"cost per point of gain: ${SPENT / gain_points:.4f}")
else:
    print("no gain. the spend bought you a negative result, which is also information.")

round 0: 25%
round 1: 50%  KEPT
round 2: 62%  KEPT
round 3: 62%  reverted
round 4: 25%  reverted
round 5: 25%  reverted
round 6: 25%  reverted
round 7: 62%  reverted
round 8: 25%  reverted
round 9: 62%  reverted
round 10: 25%  reverted

spend:        $0.0309
test gain:    +25 points
cost per point of gain: $0.0012


## Part VII: Beyond the Toy

You've built a working scale model of the systems this field's billions are chasing. The
ladders worth climbing next:

- **More mutation surface.** You evolved a prompt; the next rung is evolving the *harness*
  (retry policies, validators, context management), then skill files, then weights.
- **More serious tasks.** Twelve warehouse questions taught the mechanics. The production
  version is a containerized task with a deterministic test, an oracle solution, and a time limit.
- **More honest statistics.** Run everything three times, report the spread, not the best run.

And the one rule that never bends: **the loop never edits its own judge.** Not the task list,
not the checker, not the auditor. Every spectacular failure in the record is a system that
found a path to its own scoreboard.

The loop was never the hard part. The judge is.